# Assignment 1: N-Gram Language Models

In lecture, we defined language modeling as the task of predicting the next word in a sequence given the previous words. In this assignment, we will focus on the related problem of predicting the next character in a sequence given the previous characters.

The learning goals of this assignment are to:

- Understand how to compute language model probabilities using maximum likelihood estimation.
- Implement basic smoothing, back-off and interpolation.
- Have fun using a language model to probabilistically generate texts!
- Use a set of language models to perform text classification.

## Submission Details

Make sure to submit this python notebook along with any necessary text files needed to run your code to Brightspace by **10/05/2026 11:59pm EST**.

In [10]:
import math, random
from collections import defaultdict

In [11]:
################################################################################
# Utility Functions
################################################################################

COUNTRY_CODES = ['af', 'cn', 'de', 'fi', 'fr', 'in', 'ir', 'pk', 'za']

def start_pad(n):
    ''' Returns a padding string of length n to append to the front of text
        as a pre-processing step to building n-grams '''
    return '~' * n

def create_ngram_model(model_class, path, n=2, k=0):
    ''' Creates and returns a new n-gram model trained on the city names
        found in the path file '''
    model = model_class(n, k)
    with open(path, encoding='utf-8', errors='ignore') as f:
        model.update(f.read())
    return model

def create_ngram_model_lines(model_class, path, n=2, k=0):
    ''' Creates and returns a new n-gram model trained on the city names
        found in the path file '''
    model = model_class(n, k)
    with open(path, encoding='utf-8', errors='ignore') as f:
        for line in f:
            model.update(line.strip())
    return model

## Part 1: Generating N-Grams (4 pts)


Write a function `ngrams(n, text)` that produces a list of all n-grams of size `n` from the given `text`.  Each n-gram should consist of a 2-element tuple `(context, char)` where `context` is the `n`-length string comprised of the `n` charachters preceding `char`.

Make sure to pad the beginning of the sentence with `n` ~ charachters (I have included a function `start_pad` above for you to use).  If `n=0`, you will always have `context = ""`.  Assume that `n>= 0`.

Make sure to test your function with some examples.

e.g.
```python
>>> ngrams(1, 'abc')
[('~', 'a'), ('a', 'b'), ('b', 'c')]

>>> ngrams(2, 'abc')
[('~~', 'a'), ('~a', 'b'), ('ab', 'c')]
```

In [12]:
def ngrams(n, text):
    ''' Returns the ngrams of the text as tuples where the first element is
        the length-n context and the second is the character '''
    ### TODO ###
    padded_text = start_pad(n) + text
    result = []
    for i in range(len(text)):
        context = padded_text[i:i+n]
        char = padded_text[i+n]
        result.append((context, char))
    return result    

## Part 2: Creating an N-gram Model (20 pts)

You will build a simple n-gram language model that can be used to generate some random text that resembles a source document. Your use of external code should be limited to built-in Python modules, which excludes, for example, NumPy and NLTK.

In the `NgramModel` class, I have included an initialization method `__init__(self, n, k)` which stores the order `n` of the model and initializes any necessary internal variables.

1. Write a method `get_vocab(self)` that returns the vocab (this is the set of all characters used by this model).

2. Write a method `update(self, text)` which computes the n-grams for the input sentence and updates the internal counts.
3. Also write a method `prob(self, context, char)` which accepts an `n`-length string representing a context and a character, and returns the probability of that character occuring, given the preceding context. If you encounter a novel `context`, the probability of any given `char` should be $1/V$ where $V$ is the size of the vocab.

Make sure to test the functionality of your code.

```python
 >>> m = NgramModel(1, 0)
 >>> m.update('abab')
 >>> m.get_vocab()
 {'b', 'a'}
 >>> m.update('abcd')
 >>> m.get_vocab()
 {'b', 'a', 'c', 'd'}
 >>> m.prob('a', 'b')
 1.0
 >>> m.prob('~', 'c')
 0.0
 >>> m.prob('b', 'c')
 0.5
```


In [13]:
class NgramModel(object):
    ''' A basic n-gram model using add-k smoothing '''

    def __init__(self, n, k):
        self.n = n
        self.k = k
        self.vocab = set()
        self.context_counts = defaultdict(lambda:0)
        self.sequence_counts = defaultdict(lambda:0)

    def get_vocab(self):
        ''' Returns the set of characters in the vocab '''
        return self.vocab

    def update(self, text):
        ''' Updates the model n-grams based on text '''
        for context, char in ngrams(self.n, text):
            self.vocab.add(char)
            self.context_counts[context] += 1
            self.sequence_counts[(context, char)] += 1

    def prob(self, context, char):
        ''' Returns the probability of char appearing after context (without smoothing) '''
        if self.context_counts[context] == 0:
            return 1 / len(self.vocab)
        return self.sequence_counts[(context, char)] / self.context_counts[context]

In [14]:
m = NgramModel(1, 0)
m.update('abab')
print(m.get_vocab())
m.update('abcd')
print(m.get_vocab())
print(m.prob('a', 'b'))
print(m.prob('~', 'c'))
print(m.prob('b', 'c'))

{'a', 'b'}
{'d', 'c', 'a', 'b'}
1.0
0.0
0.5


I have provided a method `random_char(self, context)` which returns a random character according to the probability distribution determined by the given context. Specifically, let $V = \{w_1, w_2, \ldots, w_n\}$ be the vocab, sorted according to Python's natural lexicographic ordering, and let $0\leq r \leq 1$ be a random number between $0$ and $1$. The method returns a character $w_i$ such that
$$\sum_{j=1}^{i-1}P(w_j|\text{context}) \leq r < \sum_{j=1}^iP(w_j|\text{context})$$

In [15]:
def random_char(self, context):
    ''' Returns a random character based on the given context and the
        n-grams learned by this model '''
    r = random.random()
    pre_sum = 0
    for i, char in enumerate(sorted(self.vocab)):
        post_sum = pre_sum + self.prob(context, char)
        if pre_sum <= r < post_sum:
            return char
        pre_sum = post_sum

NgramModel.random_char = random_char

In [29]:
### Feel free to uncomment the following code to see how the method works ###
m = NgramModel(0, 0)
m.update('abab')
m.update('abcd')
random.seed(1)
[m.random_char('') for i in range(25)]

['a',
 'c',
 'c',
 'a',
 'b',
 'b',
 'b',
 'c',
 'a',
 'a',
 'c',
 'b',
 'c',
 'a',
 'b',
 'b',
 'a',
 'd',
 'd',
 'a',
 'a',
 'b',
 'd',
 'b',
 'a']

4. Implement the method `random_text(self, length)` which returns a string of characters chosen at random using the supplied `random_char(self, context)` method.  Your starting context should always be `n` ~ characters, and the context should be updated as characters are generated. If `n=0`, your context should always be the empty string. You should continue generating characters until you've produced the specified number of random characters, then return the full string.

Once again, be sure to test your code.  You should be able to achieve something like the following

```python
 >>> m = NgramModel(1, 0)
 >>> m.update('abab')
 >>> m.update('abcd')
 >>> random.seed(1)
 >>> m.random_text(25)
 abcdbabcdabababcdddabcdba
```

In [17]:
def random_text(self, length):
    ''' Returns text of the specified character length based on the
        n-grams learned by this model '''
    context = start_pad(self.n)
    result = []
    for _ in range(length):
        char = self.random_char(context)
        result.append(char)
        if self.n > 0:
            context = context[1:] + char
    return ''.join(result)

NgramModel.random_text = random_text

m = NgramModel(1, 0)
m.update('abab')
m.update('abcd')
random.seed(1)
print(m.random_text(25))

abcdbabcdabababcdddabcdba


### Writing Shakespeare

Now you're ready to train your first language model!  First let's grab some text:

In [18]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-10-04 19:26:39--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.1’

input.txt.1         100%[===================>]   1.06M  --.-KB/s    in 0.03s   

2026-10-04 19:26:39 (37.6 MB/s) - ‘input.txt.1’ saved [1115394/1115394]



Try generating some Shakespeare with different order n-gram models. You should try running the following commands:

In [19]:
m = create_ngram_model(NgramModel, 'input.txt', 2)
m.random_text(250)

"Fir, ad le the you, suere our nall blove my he any hattles my ledo\norsend ifin mince: han-lover th I\nmadebach for lambe viern Gaught,\nbut sione guill win!\nTalt?\nOur he re and groad, thathress,' th: appy heretten I witang niefter:\nTo racrague che why,"

In [20]:
m = create_ngram_model(NgramModel, 'input.txt', 3)
m.random_text(250)

"First Creter wrone, limble Watchese here hear I am I fors! may ture my threethis parder at men;\nBut lience their\nand speak mers oddes, unspirity: if des righs.\nWhichmannot worse have of good keeping ere.\nGremove's sequity mine o' there recaused you s"

In [21]:
m = create_ngram_model(NgramModel, 'input.txt', 4)
m.random_text(250)

"First Citizens:\nThe justice: farm:\nShe lord:\nLord, and armour not repent durst; and compassage break'st\nThat our protes of mind;\nNot put\nTo her to take in thank than thou not at hear is their shrewd.\n\nQUEEN ELIZABETH:\nI given in here's is times and h"

In [22]:
m = create_ngram_model(NgramModel, 'input.txt', 7)
m.random_text(250)

"First Citizen:\nAy, worthy man: make trial, marshal's true; I hear this ring.\nHow say you\nhad not upon recover all they kiss to chide not; for your knowledge,\nI never standards, set down--\nAs best beware; thou shalt to this discharged me a graver step"

After generating a bunch of short passages, do you notice anything? Write down some of your observations about the language model.

#### Observations

- With `n=2` the output is mostly gibberish: it has Shakespeare-like letter patterns and the speaker-label layout (names followed by a colon and a newline), but most "words" are not real words.
- With `n=3` and `n=4` more real words appear, and short phrases start to look like English, although sentences still make no sense.
- With `n=7` the text is mostly real words and phrases, and often long stretches are copied straight from the play, because most 7-character contexts have only one or two continuations in the training data.
- Higher orders therefore give more fluent text but less novelty (they memorize), while lower orders give more novelty but less structure. Higher-order models also need far more memory because the number of distinct contexts grows quickly.
- Unsmoothed models assign zero probability to any unseen sequence, so they cannot score new text well.

## Part 3: Perplexity, Smoothing, and Interpolation (6 pts)

### Perplexity
How do we know whether our language model is good?  As we talked about in lecture, we use an intrinsic evaluation method called **perplexity**.

In this part of the assignment, you will implement the `perplexity(self, text)` method.  A few things to keep in mind:
- Numeric underflow will most likely be a problem, so consider using logs (as discussed in lecture)
- Perplexity is undefined if the language model assigns any zero probabilities to the test set. In that case your code should return positive infinity - `float('inf')`.
- On your unsmoothed models, you'll definitely get some zero probabilities for the test set. To test you code, you should try computing perplexity on the training set, and you should compute perplexity for your language models that use smoothing and interpolation.

Be sure to test your code

```python
>>> m = NgramModel(1, 0)
>>> m.update('abab')
>>> m.update('abcd')
>>> m.perplexity('abcd')
1.189207115002721
>>> m.perplexity('abca')
inf
>>> m.perplexity('abcda')
1.515716566510398
```

In [23]:
def perplexity(self, text):
    ''' Returns the perplexity of text based on the n-grams learned by
        this model '''
    pairs = ngrams(self.n, text)
    log_prob = 0
    for context, char in pairs:
        p = self.prob(context, char)
        if p == 0:
            return float('inf')
        log_prob += math.log(p)
    return math.exp(-log_prob / len(pairs))

NgramModel.perplexity = perplexity

m = NgramModel(1, 0)
m.update('abab')
m.update('abcd')
print(m.perplexity('abcd'))
print(m.perplexity('abca'))
print(m.perplexity('abcda'))

1.189207115002721
inf
1.515716566510398


Discuss the perplexity for text that is similar and different from Shakespeare's plays.  

You are provided two dev text files, a New York Times article and several of Shakespeare's sonnets.  Though you are free to experiment with any text of your choosing.

In [24]:
# download and unzip the data
!wget https://computational-linguistics-class.org/homework/ngram-lms/test_data.zip
!unzip test_data.zip

--2026-10-04 19:26:41--  https://computational-linguistics-class.org/homework/ngram-lms/test_data.zip
Resolving computational-linguistics-class.org (computational-linguistics-class.org)... 185.199.110.153
Connecting to computational-linguistics-class.org (computational-linguistics-class.org)|185.199.110.153|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6998 (6.8K) [application/x-zip-compressed]
Saving to: ‘test_data.zip.1’

test_data.zip.1     100%[===================>]   6.83K  --.-KB/s    in 0.001s  

2026-10-04 19:26:42 (7.78 MB/s) - ‘test_data.zip.1’ saved [6998/6998]

/bin/bash: line 1: unzip: command not found


In [25]:
import zipfile
with zipfile.ZipFile('test_data.zip') as z:
    z.extractall()

with open('nytimes_article.txt', encoding='utf-8', errors='ignore') as f:
    nyt_text = f.read()
with open('shakespeare_sonnets.txt', encoding='utf-8', errors='ignore') as f:
    sonnet_text = f.read()

with open('input.txt', encoding='utf-8', errors='ignore') as f:
    train_text = f.read()

for n in [1, 2, 3, 4]:
    m = create_ngram_model(NgramModel, 'input.txt', n)
    print(n, 'train:', m.perplexity(train_text[:20000]), 'sonnets:', m.perplexity(sonnet_text), 'nytimes:', m.perplexity(nyt_text))

1 train: 11.982738612333367 sonnets: inf nytimes: inf
2 train: 7.0570686792451065 sonnets: inf nytimes: inf
3 train: 4.519350270800753 sonnets: inf nytimes: inf
4 train: 3.4215238742699476 sonnets: inf nytimes: inf


### Smoothing
Laplace Smoothing is described in section 3.5.1 of the [textbook](https://web.stanford.edu/~jurafsky/slp3/).  It adds 1 to each count, and since there are $V$ characters in the vocabulary and each one was incremented, we adjust the denominator to take this into account as well.

$$P_{\text{Laplace}}(w_i) = \frac{Count(w_i) + 1}{N + |V|}$$

A variant of Laplace smoothing is called Add-k smoothing or Add-epsilon smoothing. This is described in section 3.5.2 of the textbook. Update your implementation of the `prob(self, context, char)` method from Part 1 to include add-k smoothing.

Be sure to test your code

```python
>>> m = NgramModel(1, 1)
>>> m.update('abab')
>>> m.update('abcd')
>>> m.prob('a', 'a')
0.14285714285714285
>>> m.prob('a', 'b')
0.5714285714285714
>>> m.prob('c', 'd')
0.4
>>> m.prob('d', 'a')
0.25
```


In [26]:
def prob(self, context, char):
    ''' Returns the probability of char appearing after context with add-k smoothing '''
    if self.context_counts[context] == 0:
        return 1 / len(self.vocab)
    numerator = self.sequence_counts[(context, char)] + self.k
    denominator = self.context_counts[context] + self.k * len(self.vocab)
    return numerator / denominator

NgramModel.prob = prob

m = NgramModel(1, 1)
m.update('abab')
m.update('abcd')
print(m.prob('a', 'a'))
print(m.prob('a', 'b'))
print(m.prob('c', 'd'))
print(m.prob('d', 'a'))

0.14285714285714285
0.5714285714285714
0.4
0.25


### Interpolation (Optional)

The idea of interpolation is to calculate the higher order n-gram probabilities also combining the probabilities for lower-order n-gram models. Like smoothing, this helps us avoid the problem of zeros if we haven't observed the longer sequence in our training data.

$$P_{\text{interpolation}}(w_i|w_{i-2}w_{i-1}) = \lambda_1P(w_i|w_{i-2}w_{i-1}) + \lambda_2P(w_i|w_{i-1}) + \lambda_3P(w_i)$$

Below is another class definition, `NGramModelWithInterpolation` that extends the previously defined class `NGramModel`.  If you've written your code robustly, you should only need to override the following functions
- `get_vocab(self)`
- `update(self, text)`
- `prob(self, context, char)`

The value of `n` passed into the constructor is the highest order n-gram to be considered by the model (e.g. `n=2` will consider 3 different length n-grams: 2, 1, and 0).  Add-k smoothing should only occur when calculating the individual n-gram probabilities, not when calculating the overall interpolation probability!

By default set the lambdas to be equal weights, but you should also write a helper function that can be called to overwrite this default. Setting the lambdas in the helper function can either be done heuristically or by using a development set, but in the example code below, we've used the default.

```python
>>> m = NgramModelWithInterpolation(1, 0)
>>> m.update('abab')
>>> m.prob('a', 'a')
0.25
>>> m.prob('a', 'b')
0.75

>>> m = NgramModelWithInterpolation(2, 1)
>>> m.update('abab')
>>> m.update('abcd')
>>> m.prob('~a', 'b')
0.4682539682539682
>>> m.prob('ba', 'b')
0.4349206349206349
>>> m.prob('~c', 'd')
0.27222222222222225
>>> m.prob('bc', 'd')
0.3222222222222222
```

In [27]:
class NgramModelWithInterpolation(NgramModel):
    ''' An n-gram model with interpolation '''

    def __init__(self, n, k):
        super().__init__(n, k)
        self.models = [NgramModel(i, k) for i in range(n + 1)]
        self.lambdas = [1 / (n + 1)] * (n + 1)

    def set_lambdas(self, lambdas):
        ''' Overrides the default equal weights; weights are normalized to sum to 1 '''
        total = sum(lambdas)
        self.lambdas = [l / total for l in lambdas]

    def get_vocab(self):
        return self.models[0].get_vocab()

    def update(self, text):
        for model in self.models:
            model.update(text)
        self.vocab = self.models[0].get_vocab()

    def prob(self, context, char):
        total = 0
        for order, (model, lam) in enumerate(zip(self.models, self.lambdas)):
            sub_context = context[len(context) - order:] if order > 0 else ''
            total += lam * model.prob(sub_context, char)
        return total

m = NgramModelWithInterpolation(1, 0)
m.update('abab')
print(m.prob('a', 'a'))
print(m.prob('a', 'b'))

m = NgramModelWithInterpolation(2, 1)
m.update('abab')
m.update('abcd')
print(m.prob('~a', 'b'))
print(m.prob('ba', 'b'))
print(m.prob('~c', 'd'))
print(m.prob('bc', 'd'))

0.25
0.75
0.46825396825396826
0.43492063492063493
0.2722222222222222
0.3222222222222222


Now that you've created an N-gram model with interpolation, experiment with a few different lambdas and values of `k` and discuss their effects below.

In [28]:
for n in [2, 3]:
    for k in [0.01, 0.1, 1]:
        m = create_ngram_model(NgramModelWithInterpolation, 'input.txt', n, k)
        print(n, k, 'sonnets:', m.perplexity(sonnet_text), 'nytimes:', m.perplexity(nyt_text))

m = create_ngram_model(NgramModelWithInterpolation, 'input.txt', 3, 0.1)
for lambdas in [[0.25, 0.25, 0.25, 0.25], [0.05, 0.15, 0.3, 0.5], [0.7, 0.1, 0.1, 0.1]]:
    m.set_lambdas(lambdas)
    print(lambdas, 'sonnets:', m.perplexity(sonnet_text), 'nytimes:', m.perplexity(nyt_text))

2 0.01 sonnets: 10.221257109200163 nytimes: 13.079912902872266
2 0.1 sonnets: 10.2368606053359 nytimes: 12.982346409398595
2 1 sonnets: 10.380383721944286 nytimes: 13.020736567483656
3 0.01 sonnets: 7.895162626337655 nytimes: 10.800511827818411
3 0.1 sonnets: 8.005635702717127 nytimes: 10.848900714867328
3 1 sonnets: 8.574706596108266 nytimes: 11.449225001759816
[0.25, 0.25, 0.25, 0.25] sonnets: 8.005635702717127 nytimes: 10.848900714867328
[0.05, 0.15, 0.3, 0.5] sonnets: 6.514650809576413 nytimes: 9.447875729863762
[0.7, 0.1, 0.1, 0.1] sonnets: 12.254575949490619 nytimes: 15.389522656231907


#### Discussion of lambdas and k

- Small values of `k` (0.01 to 0.1) work best. Because interpolation already handles unseen contexts through the lower-order models, a large `k` just flattens each distribution and raises perplexity (for `n=3`: about 7.9 at `k=0.01` versus 8.6 at `k=1` on the sonnets).
- The lambdas matter more than `k`. Putting most of the weight on the highest-order model (`[0.05, 0.15, 0.3, 0.5]`) gave the lowest perplexity on both test files (about 6.5 on the sonnets and 9.4 on the New York Times article, against 7.9 and 10.8 with equal weights), because the high-order model is the most informative when its context has been seen and the lower orders act as a safety net.
- Putting most of the weight on the unigram model (`[0.7, 0.1, 0.1, 0.1]`) made perplexity much worse (about 12.3 and 15.4), because the model then mostly ignores context.
- Without smoothing, perplexity on the training text is finite and drops as `n` grows, but both the sonnets and the New York Times article give infinity because they contain character sequences never seen in training. Once smoothing is added, the sonnets (similar to the training data) score lower perplexity than the New York Times article (different domain), for example about 6.9 versus 11.1 for a trigram model with `k=1`.
- Interpolated models never return infinite perplexity, unlike the unsmoothed models, and they beat the single add-k models of the same order on out-of-domain text (the New York Times article).